# RO-060 PD and PID control: every number in the Note

The heading loop lives in `images/headkit.py`: the robot's moment of inertia (0.1 kg m²), its torque limit (0.2 N m),
the 45-degree corner, a 100 Hz controller with proportional, derivative and integral terms, and a gearbox that drags.
This notebook runs each case and prints the numbers the Note quotes. Nothing is random.

In [1]:
import sys
sys.path.insert(0, 'images')
import numpy as np
import headkit as h
deg = np.degrees

## 1. Force, torque and the moment of inertia

In [2]:
print('speed: 4 N on 8 kg ->', 4 / 8, 'm/s^2')
# a point mass m at radius r pushed by F square to the radius
m, r, F = 8.0, 0.1118, 1.0
print('as if all 8 kg sat at r =', round(np.sqrt(h.I / h.M), 4), 'm: I = 8 x r^2 =', round(8 * np.sqrt(h.I / h.M) ** 2, 4))
print('uniform 8 kg disc of radius 0.25 m: I =', 0.5 * 8 * 0.25 ** 2, 'kg m^2')
print('wheel forces 1 N forward and 1 N back, 0.1 m from the centre: torque =', 1 * 0.1 + 1 * 0.1, 'N m')
for I in (h.I, 0.2):
    al, w1, th1 = h.spin_up(inertia=I)
    print(f'I = {I}: alpha = {al} rad/s^2; after 1 s: {w1} rad/s and {th1} rad = {deg(th1):.1f} deg')
print('load 2.5 kg at 0.2 m adds', 2.5 * 0.2 ** 2, 'kg m^2')

speed: 4 N on 8 kg -> 0.5 m/s^2
as if all 8 kg sat at r = 0.1118 m: I = 8 x r^2 = 0.1
uniform 8 kg disc of radius 0.25 m: I = 0.25 kg m^2
wheel forces 1 N forward and 1 N back, 0.1 m from the centre: torque = 0.2 N m
I = 0.1: alpha = 2.0 rad/s^2; after 1 s: 2.0 rad/s and 1.0 rad = 57.3 deg
I = 0.2: alpha = 1.0 rad/s^2; after 1 s: 1.0 rad/s and 0.5 rad = 28.6 deg
load 2.5 kg at 0.2 m adds 0.10000000000000002 kg m^2


## 2. Proportional control on the heading

In [3]:
for kp in (0.1, 0.2):
    a = h.run(10, kp=kp)
    i = np.argmax(a['th'] >= h.TH_REF)
    t, p = h.peaks(a)
    j = np.argmin(a['th'][300:600]) + 300 if kp == 0.2 else np.argmin(a['th'][400:800]) + 400
    print(f'kp = {kp}: first torque {a["tau"][0]:.4f} N m; crosses 45 deg at {a["t"][i]:.2f} s turning at {a["w"][i]:.3f} rad/s,'
          f' torque there {a["tau"][i]:.4f}; peaks {np.round(t[:2], 2)} s at {np.round(deg(p[:2]), 2)} deg;'
          f' trough {deg(a["th"][j]):.2f} deg at {a["t"][j]:.2f} s')
a = h.run(10)
print('P at t = 1.0 s: heading', round(deg(a['th'][100]), 2), 'error', round(a['e'][100], 4), 'turn rate', round(a['w'][100], 3), 'torque', round(a['tau'][100], 4))

kp = 0.1: first torque 0.0785 N m; crosses 45 deg at 1.57 s turning at 0.788 rad/s, torque there -0.0001; peaks [3.14 9.42] s at [90.35 91.07] deg; trough -0.71 deg at 6.28 s
kp = 0.2: first torque 0.1571 N m; crosses 45 deg at 1.11 s turning at 1.117 rad/s, torque there -0.0004; peaks [2.22 6.66] s at [90.5  91.52] deg; trough -1.01 deg at 4.44 s
P at t = 1.0 s: heading 38.1 error 0.1204 turn rate 1.103 torque 0.0241


## 3. PD control

In [4]:
for kd in (0.0, 0.1, h.KD, 0.6):
    a = h.run(15, kd=kd)
    print(f'kd = {kd}: highest heading {deg(a["th"].max()):.2f} deg, within 1 deg of 45 for good after {h.within(a)} s')
print('2 sqrt(kp I) =', round(2 * np.sqrt(h.KP * h.I), 4))
a = h.run(10, kd=h.KD)
for t in (1.0, 2.0, 3.0):
    k = int(round(t / h.DT))
    print(f't = {t}: heading {deg(a["th"][k]):.2f} deg, e {a["e"][k]:.4f}, de/dt {a["de"][k]:.4f}, turn rate {a["w"][k]:.4f},'
          f' P {a["p"][k]:.4f}, D {a["d"][k]:.4f}, torque {a["tau"][k]:.4f}')

kd = 0.0: highest heading 92.57 deg, within 1 deg of 45 for good after None s
kd = 0.1: highest heading 58.82 deg, within 1 deg of 45 for good after 7.62 s
kd = 0.283: highest heading 45.00 deg, within 1 deg of 45 for good after 4.07 s
kd = 0.6: highest heading 44.76 deg, within 1 deg of 45 for good after 10.94 s
2 sqrt(kp I) = 0.2828
t = 1.0: heading 18.78 deg, e 0.4576, de/dt -0.3827, turn rate 0.3818, P 0.0915, D -0.1083, torque -0.0168
t = 2.0: heading 34.89 deg, e 0.1764, de/dt -0.1840, turn rate 0.1832, P 0.0353, D -0.0521, torque -0.0168
t = 3.0: heading 41.59 deg, e 0.0595, de/dt -0.0673, turn rate 0.0669, P 0.0119, D -0.0190, torque -0.0071


## 4. A dragging gearbox: PD stops short, PID does not

In [5]:
a = h.run(30, tau_d=h.TAU_DRAG)
t, p = h.peaks(a)
print('P only with drag: peaks', np.round(deg(p[:3]), 2), 'deg; swings about', round(deg(h.TH_REF - 0.02 / h.KP), 2), 'deg')
c = h.run(30, kd=h.KD, tau_d=h.TAU_DRAG)
print('PD with drag: ends at', round(deg(c['th'][-1]), 2), 'deg, error', round(c['e'][-1], 4), 'rad =', round(deg(c['e'][-1]), 2), 'deg; P torque', round(c['p'][-1], 4))
d = h.run(30, kd=h.KD, ki=h.KI, tau_d=h.TAU_DRAG)
t, p = h.peaks(d)
print('PID with drag: peak', round(deg(p[0]), 2), 'deg at', t[0], 's; within 1 deg after', h.within(d), 's; ends', round(deg(d['th'][-1]), 3))
for tt in (1, 2, 3, 5, 10, 30):
    k = int(round(tt / h.DT))
    print(f't = {tt}: heading {deg(d["th"][k]):.2f}, e {d["e"][k]:.4f}, sum {d["integ"][k]:.4f} rad s, I term {d["i"][k]:.4f}, P {d["p"][k]:.4f}, D {d["d"][k]:.4f}')
print('sum needed: 0.02 / 0.05 =', 0.02 / 0.05)

P only with drag: peaks [78.98 79.87 80.78] deg; swings about 39.27 deg
PD with drag: ends at 39.27 deg, error 0.1 rad = 5.73 deg; P torque 0.02


PID with drag: peak 51.78 deg at 4.74 s; within 1 deg after 10.63 s; ends 45.0
t = 1: heading 18.24, e 0.4670, sum 0.6620 rad s, I term 0.0331, P 0.0934, D -0.1148
t = 2: heading 37.35, e 0.1335, sum 0.9471 rad s, I term 0.0474, P 0.0267, D -0.0709
t = 3: heading 47.37, e -0.0413, sum 0.9808 rad s, I term 0.0490, P -0.0083, D -0.0315
t = 5: heading 51.73, e -0.1174, sum 0.7834 rad s, I term 0.0392, P -0.0235, D 0.0019
t = 10: heading 46.37, e -0.0239, sum 0.4387 rad s, I term 0.0219, P -0.0048, D 0.0033
t = 30: heading 45.00, e -0.0000, sum 0.4000 rad s, I term 0.0200, P -0.0000, D -0.0000
sum needed: 0.02 / 0.05 = 0.39999999999999997


In [6]:
a = h.run(30, kd=h.KD, ki=h.KI)
print('PID without drag: peak', round(deg(a['th'].max()), 2), 'deg; within 1 deg after', h.within(a), 's')

PID without drag: peak 55.09 deg; within 1 deg after 10.950000000000001 s


## 5. Too much integral

In [7]:
for ki in (0.0, 0.05, 0.1, 0.2, 0.4, 0.6):
    a = h.run(40, kd=h.KD, ki=ki, tau_d=h.TAU_DRAG)
    t, p = h.peaks(a)
    print(f'ki = {ki}: highest {deg(a["th"].max()):.2f} deg, within 1 deg after {h.within(a)} s, end {deg(a["th"][-1]):.2f} deg')
a = h.run(40, kd=h.KD, ki=0.6, tau_d=h.TAU_DRAG)
t, p = h.peaks(a)
print('ki = 0.6: successive peaks', np.round(deg(p[:4]), 1), 'deg at', np.round(t[:4], 2), 's')

ki = 0.0: highest 39.27 deg, within 1 deg after None s, end 39.27 deg


ki = 0.05: highest 51.78 deg, within 1 deg after 10.63 s, end 45.00 deg


ki = 0.1: highest 58.91 deg, within 1 deg after 11.07 s, end 45.00 deg


ki = 0.2: highest 68.73 deg, within 1 deg after 14.77 s, end 45.00 deg


ki = 0.4: highest 81.28 deg, within 1 deg after 38.51 s, end 44.58 deg
ki = 0.6: highest 307.49 deg, within 1 deg after None s, end -1018.76 deg


ki = 0.6: successive peaks [89.8 91.7 93.6 95.8] deg at [ 2.2   6.52 10.84 15.16] s


## 6. The derivative on a noisy heading

In [8]:
sigma = 0.005
rng = np.random.default_rng(0)
n = rng.normal(0, sigma, 100000)
de = np.diff(n) / h.DT
print('heading noise', sigma, 'rad =', round(deg(sigma), 2), 'deg; differenced every 10 ms:', round(de.std(), 3), 'rad/s; x kd:', round(h.KD * de.std(), 3), 'N m')
print('formula: sqrt(2) x 0.005 / 0.01 =', round(np.sqrt(2) * sigma / h.DT, 3))

heading noise 0.005 rad = 0.29 deg; differenced every 10 ms: 0.708 rad/s; x kd: 0.2 N m
formula: sqrt(2) x 0.005 / 0.01 = 0.707


## 7. The same integral on the speed loop of RO-059

In [9]:
M, B, DT, D_RAMP = 8.0, 2.0, 0.01, 8 * 9.81 * 0.01
for ki in (0.0, 10.0):
    v, x, s_ = 0.5, 0.0, 0.0
    vs = []
    for k in range(3001):
        e = 0.5 - v
        s_ += e * DT
        u = min(max(B * 0.5 + 20 * e + ki * s_, -4), 4)
        d = D_RAMP if x >= 2 - 1e-9 else 0.0
        vs.append(v)
        x += v * DT
        v += DT * (u - B * v - d) / M
    vs = np.array(vs)
    print(f'ki = {ki}: lowest {vs.min():.3f} m/s, at 10 s {vs[1000]:.4f}, at 30 s {vs[-1]:.4f} m/s')

ki = 0.0: lowest 0.464 m/s, at 10 s 0.4643, at 30 s 0.4643 m/s
ki = 10.0: lowest 0.472 m/s, at 10 s 0.4980, at 30 s 0.5000 m/s
